# 03 - Interpretacion de Regimenes, Auditoria Forense de Anomalias y Prediccion en Produccion

## Curso de Machine Learning: Modelos No Supervisados

En la practica financiera y bancaria, aplicar DBSCAN produce un doble valor estrategico que ningun algoritmo particional clasico ofrece:
1. **Comprension Estructural de Regimenes ($C_0, C_1, \dots$)**: Permite caracterizar la anatomia financiera de cada segmento saludable del mercado.
2. **Auditoria Forense de Anomalias ($C_{-1}$)**: El subconjunto de observaciones etiquetadas como `-1` constituye una bandeja de alertas prioritarias para el equipo de **Riesgo de Credito, Prevencion de Lavado de Dinero (AML) o Vigilancia de Mercado**.

Ademas, en este cuaderno resolveremos un reto fundamental de **Ingenieria de Machine Learning (MLOps)**: dado que `DBSCAN` en Scikit-Learn no cuenta con el metodo `.predict(X_nuevos)` porque es un algoritmo transductivo, implementaremos un **Clasificador Inductivo Basado en Puntos Nucleo** y un **Arbol de Decision Sustituto** para poder clasificar nuevas emisiones o transacciones en milisegundos dentro de un entorno productivo.

---

### Contenidos
1. Caracterizacion Robusta de Regimenes: Medianas vs. Medias (Por que no usar Centroides Simples)
2. Mapa de Calor de Perfiles Robustos ($Z$-Scores Basados en Mediana e IQR)
3. Distribuciones Comparativas mediante Diagramas de Caja / Violin (Regimenes vs. Anomalias `-1`)
4. Auditoria Forense del Cluster `-1`: Explicando Por Que un Activo es Anomalo
5. MLOps para DBSCAN: Implementacion de `.predict()` para Nuevos Datos usando Puntos Nucleo (`KNeighborsClassifier`)
6. Reglas de Politica de Credito mediante Arbol de Decision Sustituto
7. Visualizacion Topologica en 2D con PCA Destacando Nucleos, Fronteras y Alertas
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Generacion y ajuste del modelo DBSCAN calibrado sobre el portafolio de credito institucional
def generar_dataset_credito_institucional(seed=42):
    rng = np.random.default_rng(seed)
    r0 = np.column_stack([
        rng.normal(1.4, 0.35, 220).clip(0.3, 2.5),
        rng.normal(11.5, 2.0, 220).clip(6.5, 18.0),
        45 + 18 * (rng.normal(1.4, 0.35, 220).clip(0.3, 2.5) ** 1.5) + rng.normal(0, 8, 220),
        rng.normal(16.0, 2.5, 220).clip(10.0, 23.0),
        rng.normal(4.2, 0.9, 220).clip(1.8, 7.5),
        rng.normal(14.5, 2.8, 220).clip(7.0, 22.0)
    ])
    r1 = np.column_stack([
        rng.normal(0.45, 0.25, 160).clip(0.0, 1.2),
        rng.normal(24.0, 3.5, 160).clip(15.0, 35.0),
        rng.normal(65.0, 12.0, 160).clip(35.0, 100.0),
        rng.normal(31.0, 3.8, 160).clip(22.0, 42.0),
        rng.normal(2.5, 0.6, 160).clip(1.0, 4.5),
        rng.normal(29.5, 4.2, 160).clip(20.0, 42.0)
    ])
    r2 = np.column_stack([
        rng.normal(4.3, 0.55, 140).clip(3.0, 5.8),
        rng.normal(2.8, 0.6, 140).clip(1.4, 4.5),
        260 + 25 * (rng.normal(4.3, 0.55, 140).clip(3.0, 5.8) ** 1.4) + rng.normal(0, 25, 140),
        rng.normal(42.0, 4.5, 140).clip(32.0, 54.0),
        rng.normal(14.5, 2.2, 140).clip(9.0, 21.0),
        rng.normal(6.5, 3.0, 140).clip(-2.0, 14.0)
    ])
    anomalias = np.column_stack([
        rng.uniform(7.5, 14.0, 30),
        rng.uniform(-3.5, 0.9, 30),
        rng.uniform(1100.0, 3800.0, 30),
        rng.uniform(70.0, 145.0, 30),
        rng.uniform(38.0, 95.0, 30),
        rng.uniform(-45.0, -12.0, 30)
    ])
    X_total = np.vstack([r0, r1, r2, anomalias])
    perm = rng.permutation(len(X_total))
    columnas = [
        'deuda_neta_ebitda', 'cobertura_intereses_icr', 'spread_cds_5y_bps',
        'volatilidad_implicita_30d', 'bid_ask_spread_bps', 'retorno_roe_pct'
    ]
    tickers = [f'BOND_{i:03d}' for i in range(1, len(X_total) + 1)]
    return pd.DataFrame(X_total[perm], index=tickers, columns=columnas)

df_credito = generar_dataset_credito_institucional()
scaler = RobustScaler()
X_rob = scaler.fit_transform(df_credito)

dbscan = DBSCAN(eps=0.85, min_samples=10)
df_credito['Regimen'] = dbscan.fit_predict(X_rob)


## 1. Caracterizacion Robusta de Regimenes (Medianas por Grupo)

A diferencia de K-Means, DBSCAN no almacena un atributo `.cluster_centers_` porque un cluster de densidad arbitraria (como una media luna o un anillo) podria tener su centro de masa fuera del propio cluster.

Sin embargo, podemos caracterizar cada regimen financiero calculando el **vector de medianas** (medoid sintetico) tanto en las unidades financieras originales como en el espacio escalado `RobustScaler`.

In [ ]:
# Tabla resumen de medianas en unidades financieras originales incluyendo el grupo de Anomalias (-1)resumen_medianas = df_credito.groupby('Regimen').median()resumen_medianas['N_Emisiones'] = df_credito.groupby('Regimen').size()nombres_regimenes = {    -1: '[-1] ALERTA: Distressed / Anomalias',     0: '[0] Regimen Conservador / Investment Grade',     1: '[1] Regimen Crecimiento / High-Tech Liquido',     2: '[2] Regimen Apalancado / High-Yield Ciclico'}resumen_medianas.index = [nombres_regimenes.get(i, f'Regimen {i}') for i in resumen_medianas.index]print("=" * 85)print("PERFIL FINANCIERO MEDIANO POR REGIMEN Y GRUPO DE ANOMALIAS (UNIDADES REALES)")print("=" * 85)print(resumen_medianas.T.round(2))

In [ ]:
# Mapa de calor en espacio RobustScaler para comparar perfilesdf_rob_temp = pd.DataFrame(X_rob, columns=df_credito.columns[:-1])df_rob_temp['Regimen'] = df_credito['Regimen'].valuesmedianas_rob = df_rob_temp.groupby('Regimen').median()medianas_rob.index = [nombres_regimenes.get(i, f'Regimen {i}') for i in medianas_rob.index]plt.figure(figsize=(11, 5.5))sns.heatmap(medianas_rob.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.8,            cbar_kws={'label': 'Unidades de Rango Intercuartilico respecto a la Mediana Global'})plt.title('Huella Financiera de los Regimenes y del Grupo Distressed [-1] (Espacio RobustScaler)',           fontsize=12.5, fontweight='bold')plt.tight_layout()plt.show()

## 2. Auditoria Forense del Cluster `-1`: Explicando Cada Anomalia

Cuando un analista de riesgos recibe una lista de 30 bonos marcados como `Regimen = -1`, su primera pregunta es: **"¿Por que el modelo marco especificamente al bono `BOND_XXX` como anomalia y que variable causo la alerta?"**

Podemos construir un **Motor de Explicabilidad Forense para DBSCAN** siguiendo tres pasos matematicos:
1. Para cada anomalia $x_{out} \in C_{-1}$, buscamos el **Punto Nucleo mas cercano** $p^* \in X_{core}$ en el espacio escalado.
2. Identificamos a que regimen pertenece ese punto nucleo ($C_k$) como el "regimen de referencia esperado".
3. Calculamos la contribucion porcentual de cada variable $j$ a la distancia cuadrada que separo a la anomalia de dicho nucleo:
   $$\text{Contribucion}_j(x_{out}) = \frac{(x_{out, j} - p^*_j)^2}{\sum_{m=1}^{p} (x_{out, m} - p^*_m)^2} \times 100\%$$

In [ ]:
# Implementacion del Motor de Explicabilidad Forense para las Anomalias (-1)columnas_feat = [c for c in df_credito.columns if c != 'Regimen']core_indices = dbscan.core_sample_indices_X_cores = X_rob[core_indices]labels_cores = dbscan.labels_[core_indices]nn_forense = NearestNeighbors(n_neighbors=1).fit(X_cores)mask_out = (df_credito['Regimen'] == -1)X_outliers_rob = X_rob[mask_out]tickers_outliers = df_credito.index[mask_out]dist_a_nucleo, idx_nucleo_cercano = nn_forense.kneighbors(X_outliers_rob)idx_nucleo_cercano = idx_nucleo_cercano.ravel()# Calculo de contribuciones por variable para el Top 5 de anomalias mas severasorden_severidad = np.argsort(dist_a_nucleo.ravel())[::-1]reporte_forense = []for idx in orden_severidad[:5]:    ticker = tickers_outliers[idx]    vec_out = X_outliers_rob[idx]    vec_core = X_cores[idx_nucleo_cercano[idx]]    diff_sq = (vec_out - vec_core) ** 2    contrib_pct = (diff_sq / diff_sq.sum()) * 100    top_var_idx = np.argmax(contrib_pct)        reporte_forense.append({        'Ticker_Emision': ticker,        'Dist_al_Nucleo': round(dist_a_nucleo.ravel()[idx], 2),        'Variable_Critica_1': f"{columnas_feat[top_var_idx]} ({contrib_pct[top_var_idx]:.1f}%)",        'Spread_CDS (bps)': round(df_credito.loc[ticker, 'spread_cds_5y_bps'], 1),        'Deuda/EBITDA': round(df_credito.loc[ticker, 'deuda_neta_ebitda'], 2),        'Bid_Ask (bps)': round(df_credito.loc[ticker, 'bid_ask_spread_bps'], 1)    })df_forense = pd.DataFrame(reporte_forense)print("=" * 90)print("REPORTE DE AUDITORIA FORENSE: TOP 5 EMISIONES MAS ANOMALAS Y CAUSA RAIZ DE LA ALERTA")print("=" * 90)print(df_forense.to_string(index=False))

## 3. MLOps para DBSCAN: Como Predecir sobre Nuevos Datos en Tiempo Real

Si intentas ejecutar `dbscan.predict(X_nuevos)` en Scikit-Learn, obtendras un `AttributeError: 'DBSCAN' object has no attribute 'predict'`. ¿Por que? Porque un punto nuevo podria actuar como "puente" entre dos clusters existentes y alterar retroactivamente toda la particion.

En produccion bancaria, no podemos reentrenar DBSCAN desde cero por cada nueva solicitud de credito. La **arquitectura MLOps exacta** reproduce matematicamente la regla de puntos frontera de DBSCAN:
1. Extraemos exclusivamente los **Puntos Nucleo** (`X_rob[dbscan.core_sample_indices_]`) y sus etiquetas.
2. Para cualquier nueva observacion $x_{new}$:
   - Calculamos su distancia al Punto Nucleo mas cercano.
   - Si $\text{dist}(x_{new}, p_{core}^*) \leq \varepsilon$, le asignamos el regimen de ese Punto Nucleo.
   - Si $\text{dist}(x_{new}, p_{core}^*) > \varepsilon$, lo declaramos inmediatamente como **Anomalia (`-1`)**.

In [ ]:
class DetectorRegimenesDBSCANProduccion:    """Wrapper MLOps inductivo para habilitar .predict() en tiempo real sobre un modelo DBSCAN entrenado."""    def __init__(self, scaler, dbscan_model):        self.scaler = scaler        self.eps = dbscan_model.eps        self.core_indices = dbscan_model.core_sample_indices_        self.nn_core = NearestNeighbors(n_neighbors=1, metric=dbscan_model.metric)            def fit_cores(self, X_scaled, labels):        self.X_cores = X_scaled[self.core_indices]        self.y_cores = labels[self.core_indices]        self.nn_core.fit(self.X_cores)        return self            def predict(self, X_nuevos_df):        X_num = self.scaler.transform(X_nuevos_df)        distancias, idx_cercano = self.nn_core.kneighbors(X_num)        distancias = distancias.ravel()        idx_cercano = idx_cercano.ravel()                predicciones = np.where(distancias <= self.eps, self.y_cores[idx_cercano], -1)        return predicciones, distancias# Instanciamos nuestro predictor para produccionmotor_produccion = DetectorRegimenesDBSCANProduccion(scaler, dbscan).fit_cores(X_rob, dbscan.labels_)# Simulamos la llegada de 3 nuevas emisiones de bonos al sistema en tiempo realnuevos_bonos = pd.DataFrame([    [1.3, 12.0, 68.0, 15.5, 4.0, 15.0],    # Bono Solido Tipo Investment Grade    [0.3, 25.0, 60.0, 30.0, 2.2, 31.0],    # Emisor Tecnologico Liquido    [9.8, -1.2, 2450.0, 95.0, 65.0, -28.0] # Emisor en Colapso de Liquidez (Distressed)], columns=columnas_feat, index=['NUEVO_IG_01', 'NUEVO_TECH_02', 'NUEVO_COLAPSO_03'])preds, dists = motor_produccion.predict(nuevos_bonos)nuevos_bonos['Regimen_Predicho'] = predsnuevos_bonos['Distancia_a_Nucleo'] = dists.round(3)print("Prediccion en Tiempo Real sobre Nuevas Emisiones (Out-of-Sample):")print(nuevos_bonos[['deuda_neta_ebitda', 'spread_cds_5y_bps', 'Regimen_Predicho', 'Distancia_a_Nucleo']])

## 4. Explicabilidad Ejecutiva mediante Arbol de Decision Sustituto

Finalmente, entrenamos un `DecisionTreeClassifier` de profundidad 3 sobre las unidades financieras reales para proporcionar al Comite de Inversiones las reglas explicitas que separan los 3 regimenes y la zona de alerta (`-1`).

In [ ]:
X_orig = df_credito[columnas_feat]y_reg = df_credito['Regimen']surrogate_tree = DecisionTreeClassifier(max_depth=3, random_state=42)surrogate_tree.fit(X_orig, y_reg)fidelidad = surrogate_tree.score(X_orig, y_reg)print(f"Fidelidad del Arbol Sustituto explicando a DBSCAN (incluyendo Anomalias): {fidelidad*100:.2f}%\n")print(export_text(surrogate_tree, feature_names=columnas_feat, decimals=2))plt.figure(figsize=(18, 7.5))plot_tree(surrogate_tree, feature_names=columnas_feat,          class_names=['Alerta [-1]', 'Regimen 0 (IG)', 'Regimen 1 (Tech)', 'Regimen 2 (HY)'],          filled=True, rounded=True, fontsize=9.5, precision=1)plt.title(f'Reglas de Politica Crediticia Extraidas de DBSCAN (Fidelidad = {fidelidad*100:.1f}%)',           fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

## 5. Conclusiones

### Puntos Clave:
1. **Estadisticos Robustos para Perfilamiento**: Al interpretar regimenes de DBSCAN, utiliza siempre la **mediana** por grupo en lugar de la media para mantener coherencia con `RobustScaler`.
2. **Auditoria Forense de `-1`**: Medir la distancia vectorial desde cada anomalia hacia su Punto Nucleo mas cercano permite atribuir exactamente que variable financiera disparo la alerta.
3. **Despliegue Inductivo en Produccion**: Combinar `NearestNeighbors(n_neighbors=1)` sobre los `core_sample_indices_` con el umbral $\varepsilon$ dota a DBSCAN de capacidad `.predict()` instantanea para sistemas en produccion.